# 01 Regime Detection Visual Study

## Objective
Study, visualize, and document the already-computed quantile and HMM regime labels for the thesis scope: BTCUSDT and ETHUSDT, 15-minute and 1-hour frequencies, 2020-2025.

## Input files
- Quantile regime outputs under `results/regimes/quantile`
- HMM regime outputs under `results/regimes/hmm`
- Feature files under `final_dataset/features`

## Output folder
`reports/study_notebooks`, especially `figures/regime` and `tables`.

## Thesis relevance
These figures explain what market regimes were detected and how those labels condition motif discovery. This notebook does not rerun regime detection or modify original result files.

## Analysis-only safety
This notebook never imports or calls STUMPY, STUMP/MSTUMP, HMM fitting, LoCoMotif search, or any other expensive experiment algorithm. It only reads saved result files and thesis-scope feature parquet files, then produces derived tables and figures.

**Quantile caveat.** The quantile regime outputs store `rolling_volatility_60` as the actual volatility column across all quantile method identifiers. Therefore, quantile regimes are interpreted as 60-period rolling-volatility regimes with different regime-count granularities rather than as separate 30/60/240 volatility-horizon experiments.


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd() / "HPC workflow" / "HPC_Regime_and_motif_discovery" / "notebooks" / "study"
if NOTEBOOK_DIR.exists() and str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from study_helpers import *

ensure_study_output_dirs()
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("Project root:", PROJECT_ROOT)
print("Workflow root:", WORKFLOW_ROOT)
print("Study outputs:", REPORT_ROOT)


## Load Quantile and HMM Thesis-Scope Files


In [ ]:
quantile_dir = result_path("regimes", "quantile")
hmm_dir = result_path("regimes", "hmm")

quantile_files = {
    "labels": resolve_existing_file(quantile_dir, "quantile_regime_labels.parquet"),
    "summary": resolve_existing_file(quantile_dir, "quantile_regime_summary.parquet"),
    "transitions": resolve_existing_file(quantile_dir, "quantile_transition_matrix.parquet"),
}
hmm_files = {
    "labels": resolve_existing_file(hmm_dir, "hmm_regime_labels.parquet"),
    "summary": resolve_existing_file(hmm_dir, "hmm_regime_summary.parquet"),
    "transitions": resolve_existing_file(hmm_dir, "hmm_transition_matrix.parquet"),
    "persistence": resolve_existing_file(hmm_dir, "hmm_persistence_metrics.parquet"),
}

quantile_labels = coerce_timestamp(safe_read_parquet(quantile_files["labels"]))
quantile_summary = safe_read_parquet(quantile_files["summary"])
quantile_transitions = safe_read_parquet(quantile_files["transitions"])
hmm_labels = coerce_timestamp(safe_read_parquet(hmm_files["labels"]))
hmm_summary = safe_read_parquet(hmm_files["summary"])
hmm_transitions = safe_read_parquet(hmm_files["transitions"])
hmm_persistence = safe_read_parquet(hmm_files["persistence"])

file_inventory = pd.DataFrame([
    {"dataset": f"quantile_{k}", "path": str(v), "exists": v.exists()} for k, v in quantile_files.items()
] + [
    {"dataset": f"hmm_{k}", "path": str(v), "exists": v.exists()} for k, v in hmm_files.items()
])
display_table(file_inventory)
save_table(file_inventory, "study_regime_file_inventory")


## Validate Files, Shapes, Columns, and Date Ranges


In [ ]:
def frame_schema(name, df):
    ts = timestamp_column(df)
    return {
        "name": name,
        "rows": len(df),
        "columns": len(df.columns) if df is not None else 0,
        "timestamp_column": ts,
        "min_timestamp": df[ts].min() if ts and not df.empty else pd.NaT,
        "max_timestamp": df[ts].max() if ts and not df.empty else pd.NaT,
        "useful_columns": useful_columns(df),
    }

schema = pd.DataFrame([
    frame_schema("quantile_labels", quantile_labels),
    frame_schema("quantile_summary", quantile_summary),
    frame_schema("quantile_transitions", quantile_transitions),
    frame_schema("hmm_labels", hmm_labels),
    frame_schema("hmm_summary", hmm_summary),
    frame_schema("hmm_transitions", hmm_transitions),
    frame_schema("hmm_persistence", hmm_persistence),
])
display_table(schema, 20)
save_table(schema, "study_regime_schema_validation")


## Regime Coverage Table


In [ ]:
def coverage_row(df, asset, frequency, method_name):
    scoped = filter_scope(df, asset, frequency)
    ts = timestamp_column(scoped)
    reg = regime_column(scoped)
    method_col = "regime_method" if "regime_method" in scoped.columns else None
    return {
        "asset": asset,
        "frequency": frequency,
        "method": method_name,
        "rows": len(scoped),
        "min_timestamp": scoped[ts].min() if ts and not scoped.empty else pd.NaT,
        "max_timestamp": scoped[ts].max() if ts and not scoped.empty else pd.NaT,
        "regime_methods": ", ".join(sorted(scoped[method_col].dropna().astype(str).unique())) if method_col and not scoped.empty else "",
        "regime_labels": ", ".join(sorted(scoped[reg].dropna().astype(str).unique())) if reg and not scoped.empty else "",
    }

coverage = pd.DataFrame(
    [coverage_row(quantile_labels, asset, frequency, "quantile") for asset in THESIS_SCOPE_ASSETS for frequency in THESIS_SCOPE_FREQUENCIES]
    + [coverage_row(hmm_labels, asset, frequency, "HMM") for asset in THESIS_SCOPE_ASSETS for frequency in THESIS_SCOPE_FREQUENCIES]
)
display_table(coverage, 20)
save_table(coverage, "study_regime_coverage_table")


## Regime Label Counts


In [ ]:
def plot_regime_counts(df, method_name, asset, frequency, filename):
    scoped = filter_scope(df, asset, frequency)
    if method_name.lower() == "quantile":
        selected = choose_quantile_method(scoped)
        if selected and "regime_method" in scoped.columns:
            scoped = scoped[scoped["regime_method"].astype(str) == selected]
            method_title = f"{method_name}: {selected}"
        else:
            method_title = method_name
    else:
        method_title = method_name
    reg = regime_column(scoped)
    if scoped.empty or not reg:
        print(f"No label counts available for {asset} {frequency} {method_name}")
        return
    counts = scoped[reg].astype(str).value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(10, 5))
    counts.plot(kind="bar", ax=ax, color="#4C78A8")
    ax.set_title(f"Regime label counts - {asset} {frequency} {method_title}")
    ax.set_xlabel("Regime label")
    ax.set_ylabel("Rows")
    fig.tight_layout()
    save_fig(fig, filename, FIGURE_DIRS["regime"])
    plt.show()

for asset in ["BTCUSDT", "ETHUSDT"]:
    plot_regime_counts(quantile_labels, "quantile", asset, "15m", f"study_regime_counts_{asset}_15m_quantile")
    plot_regime_counts(hmm_labels, "HMM", asset, "15m", f"study_regime_counts_{asset}_15m_hmm")


## Price and Regime Overlays


In [ ]:
def plot_price_regime_overlay(labels, method_name, asset, frequency, filename, max_points=6000):
    scoped = filter_scope(labels, asset, frequency)
    if method_name.lower() == "quantile":
        selected = choose_quantile_method(scoped)
        if selected and "regime_method" in scoped.columns:
            scoped = scoped[scoped["regime_method"].astype(str) == selected]
    ts = timestamp_column(scoped)
    reg = regime_column(scoped)
    feature = load_feature_data(asset, frequency)
    fts = timestamp_column(feature)
    if scoped.empty or feature.empty or not ts or not fts or not reg or "close" not in feature.columns:
        print(f"Cannot draw overlay for {asset} {frequency} {method_name}: missing labels, timestamps, or close.")
        return
    merged = pd.merge(
        feature[[fts, "close"]].rename(columns={fts: "timestamp"}),
        scoped[[ts, reg]].rename(columns={ts: "timestamp", reg: "regime_label"}),
        on="timestamp",
        how="inner",
    ).dropna(subset=["timestamp", "close", "regime_label"]).sort_values("timestamp")
    if merged.empty:
        print(f"No timestamp overlap for {asset} {frequency} {method_name}")
        return
    if len(merged) > max_points:
        step = int(np.ceil(len(merged) / max_points))
        merged = merged.iloc[::step].copy()
    labels_sorted = sorted(merged["regime_label"].astype(str).unique())
    color_map = {label: plt.cm.tab10(i % 10) for i, label in enumerate(labels_sorted)}
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(merged["timestamp"], merged["close"], color="0.75", linewidth=0.8, label="close")
    for label in labels_sorted:
        part = merged[merged["regime_label"].astype(str) == label]
        ax.scatter(part["timestamp"], part["close"], s=7, color=color_map[label], label=str(label), alpha=0.75)
    ax.set_title(f"{asset} {frequency} close price by {method_name} regime")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("Close price")
    ax.legend(title="Regime", ncol=min(4, len(labels_sorted)))
    fig.autofmt_xdate()
    fig.tight_layout()
    save_fig(fig, filename, FIGURE_DIRS["regime"])
    plt.show()

for asset in ["BTCUSDT", "ETHUSDT"]:
    plot_price_regime_overlay(quantile_labels, "quantile", asset, "15m", f"study_regime_{asset}_15m_quantile_close_by_regime")
    plot_price_regime_overlay(hmm_labels, "HMM", asset, "15m", f"study_regime_{asset}_15m_hmm_close_by_regime")


### Interpretation
The overlay figures show where regime labels occur on the observed price path. Dense 15-minute data is sampled only for visualization clarity; the saved regime labels themselves are not changed.


## Volatility Distribution by Regime


In [ ]:
def plot_volatility_by_regime(labels, method_name, asset, frequency, filename):
    scoped = filter_scope(labels, asset, frequency)
    if method_name.lower() == "quantile":
        selected = choose_quantile_method(scoped)
        if selected and "regime_method" in scoped.columns:
            scoped = scoped[scoped["regime_method"].astype(str) == selected]
    reg = regime_column(scoped)
    vol_col = next((c for c in ["volatility_value", "rolling_volatility_60", "rolling_vol"] if c in scoped.columns), None)
    if not vol_col:
        feature = load_feature_data(asset, frequency)
        ts = timestamp_column(scoped)
        fts = timestamp_column(feature)
        feature_vol = next((c for c in ["rolling_volatility_60", "rolling_vol"] if c in feature.columns), None)
        if ts and fts and feature_vol and reg:
            scoped = pd.merge(
                scoped[[ts, reg]].rename(columns={ts: "timestamp"}),
                feature[[fts, feature_vol]].rename(columns={fts: "timestamp"}),
                on="timestamp",
                how="inner",
            )
            vol_col = feature_vol
    if scoped.empty or not reg or not vol_col:
        print(f"No volatility column available for {asset} {frequency} {method_name}")
        return
    groups = [g[vol_col].dropna().to_numpy() for _, g in scoped.groupby(reg)]
    labels_order = [str(k) for k, _ in scoped.groupby(reg)]
    if not groups:
        print(f"No volatility observations available for {asset} {frequency} {method_name}")
        return
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.boxplot(groups, labels=labels_order, showfliers=False)
    ax.set_title(f"Volatility distribution by regime - {asset} {frequency} {method_name}")
    ax.set_xlabel("Regime label")
    ax.set_ylabel(vol_col)
    fig.tight_layout()
    save_fig(fig, filename, FIGURE_DIRS["regime"])
    plt.show()

for asset in ["BTCUSDT", "ETHUSDT"]:
    plot_volatility_by_regime(quantile_labels, "quantile", asset, "15m", f"study_regime_{asset}_15m_quantile_volatility_by_regime")
    plot_volatility_by_regime(hmm_labels, "HMM", asset, "15m", f"study_regime_{asset}_15m_hmm_volatility_by_regime")


## Transition Heatmaps


In [ ]:
q_method = choose_quantile_method(filter_scope(quantile_labels, "BTCUSDT", "15m"))
q_filters = {"asset": "BTCUSDT", "frequency": "15m"}
if q_method:
    q_filters["regime_method"] = q_method
plot_heatmap_from_table(
    quantile_transitions,
    "BTCUSDT 15m quantile transition probabilities",
    "study_regime_BTCUSDT_15m_quantile_transition_heatmap",
    FIGURE_DIRS["regime"],
    q_filters,
)
plot_heatmap_from_table(
    hmm_transitions,
    "BTCUSDT 15m HMM transition probabilities",
    "study_regime_BTCUSDT_15m_hmm_transition_heatmap",
    FIGURE_DIRS["regime"],
    {"asset": "BTCUSDT", "frequency": "15m"},
)


## Regime Duration and Persistence


In [ ]:
def segment_durations(labels, method_name):
    rows = []
    if labels is None or labels.empty:
        return pd.DataFrame()
    ts = timestamp_column(labels)
    reg = regime_column(labels)
    if not ts or not reg:
        return pd.DataFrame()
    group_cols = [c for c in ["asset", "frequency", "regime_method"] if c in labels.columns]
    for keys, part in labels.dropna(subset=[ts, reg]).sort_values(ts).groupby(group_cols, dropna=False):
        part = part.sort_values(ts).copy()
        segment_id = (part[reg].astype(str) != part[reg].astype(str).shift()).cumsum()
        for sid, seg in part.groupby(segment_id):
            rec = {col: val for col, val in zip(group_cols, keys if isinstance(keys, tuple) else (keys,))}
            rec.update({
                "method_family": method_name,
                "regime_label": seg[reg].iloc[0],
                "duration_observations": len(seg),
                "start_timestamp": seg[ts].min(),
                "end_timestamp": seg[ts].max(),
            })
            rows.append(rec)
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    freq_minutes = {"15m": 15, "1h": 60}
    out["approx_duration_hours"] = out.apply(
        lambda r: r["duration_observations"] * freq_minutes.get(str(r.get("frequency")), np.nan) / 60,
        axis=1,
    )
    return out

durations = pd.concat([
    segment_durations(quantile_labels, "quantile"),
    segment_durations(hmm_labels, "HMM"),
], ignore_index=True)
display_table(durations, 20)
save_table(durations, "study_regime_segment_durations")

summary_cols = [c for c in ["method_family", "asset", "frequency", "regime_method", "regime_label"] if c in durations.columns]
duration_summary = durations.groupby(summary_cols, dropna=False)["duration_observations"].agg(["count", "mean", "median"]).reset_index() if not durations.empty else pd.DataFrame()
display_table(duration_summary, 30)
save_table(duration_summary, "study_regime_duration_summary")

if not durations.empty:
    focus = durations[durations.get("frequency", "").astype(str).eq("15m")] if "frequency" in durations.columns else durations
    fig, ax = plt.subplots(figsize=(12, 5))
    labels_box = []
    values = []
    for key, part in focus.groupby(["method_family", "regime_label"], dropna=False):
        labels_box.append(" / ".join(map(str, key)))
        values.append(part["duration_observations"].to_numpy())
    if values:
        ax.boxplot(values, labels=labels_box, showfliers=False)
        ax.set_title("Regime segment duration distribution, 15m labels")
        ax.set_ylabel("Consecutive observations")
        ax.tick_params(axis="x", rotation=45)
        fig.tight_layout()
        save_fig(fig, "study_regime_duration_distribution_15m", FIGURE_DIRS["regime"])
        plt.show()


## Key findings
Use the tables and figures above to report which regime files are complete and which labels are available for each thesis-scope asset/frequency pair.

## Thesis-safe interpretation
Quantile regimes provide deterministic partitions of observed rolling volatility. HMM regimes provide latent probabilistic regimes inferred from multiple features when the corresponding HMM output files contain labels. Both families are used as alternative market-state labels for conditioned motif discovery.

## Limitations
The notebook only reports what exists in the result files. Missing files, empty files, or missing columns are displayed explicitly and should not be converted into numerical claims.

## Recommended figures for thesis
- `study_regime_BTCUSDT_15m_quantile_close_by_regime`
- `study_regime_BTCUSDT_15m_hmm_close_by_regime` when HMM labels are available
- `study_regime_BTCUSDT_15m_quantile_transition_heatmap`
- `study_regime_duration_distribution_15m`
